# Comprehensive Method Comparison

This notebook provides an in-depth comparison of all available imputation methods.

## Objectives

1. Compare all 16+ imputation methods
2. Evaluate performance across different data patterns
3. Understand computational trade-offs
4. Guide method selection for your use case

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import load_diabetes, load_breast_cancer

# Import all imputation methods
from imputation_showcase.imputation_methods import (
    MeanImputer,
    MedianImputer,
    KNNImputerMethod,
    MICEImputer,
    LOCFImputer,
    NOCBImputer,
    HotDeckImputer,
    rmse,
    mae,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Setup complete")

## 1. Load Test Dataset

We'll use the diabetes dataset from scikit-learn.

In [ ]:
# Load complete dataset
diabetes = load_diabetes(as_frame=True)
data_complete = diabetes.frame.iloc[:, :8]  # Use first 8 features

print(f"Dataset shape: {data_complete.shape}")
print(f"\nFeatures: {list(data_complete.columns)}")
print(f"\nDataset statistics:")
print(data_complete.describe())

## 2. Create Missing Data Patterns

We'll test three common missing data patterns:

1. **MCAR** - Missing Completely At Random (10%)
2. **MAR** - Missing At Random (15%)
3. **MNAR** - Missing Not At Random (20%)

In [ ]:
def create_mcar(data, missing_rate=0.1, seed=42):
    """Missing Completely At Random."""
    np.random.seed(seed)
    data_missing = data.copy()
    mask = np.random.rand(*data.shape) < missing_rate
    data_missing[mask] = np.nan
    return data_missing

def create_mar(data, missing_rate=0.15, seed=42):
    """Missing At Random - depends on other variables."""
    np.random.seed(seed)
    data_missing = data.copy()
    # Make missingness depend on first column
    threshold = data.iloc[:, 0].quantile(0.7)
    high_values = data.iloc[:, 0] > threshold
    for col in data.columns[1:]:
        mask = high_values & (np.random.rand(len(data)) < missing_rate)
        data_missing.loc[mask, col] = np.nan
    return data_missing

def create_mnar(data, missing_rate=0.2, seed=42):
    """Missing Not At Random - depends on the value itself."""
    np.random.seed(seed)
    data_missing = data.copy()
    for col in data.columns:
        # Higher values more likely to be missing
        threshold = data[col].quantile(0.6)
        high_values = data[col] > threshold
        mask = high_values & (np.random.rand(len(data)) < missing_rate)
        data_missing.loc[mask, col] = np.nan
    return data_missing

# Create datasets with different missing patterns
data_mcar = create_mcar(data_complete)
data_mar = create_mar(data_complete)
data_mnar = create_mnar(data_complete)

print("Missing data created:")
print(f"MCAR: {data_mcar.isna().sum().sum()} missing ({data_mcar.isna().sum().sum()/data_mcar.size*100:.1f}%)")
print(f"MAR:  {data_mar.isna().sum().sum()} missing ({data_mar.isna().sum().sum()/data_mar.size*100:.1f}%)")
print(f"MNAR: {data_mnar.isna().sum().sum()} missing ({data_mnar.isna().sum().sum()/data_mnar.size*100:.1f}%)")

In [ ]:
# Visualize missing patterns
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

datasets = [data_mcar, data_mar, data_mnar]
titles = ['MCAR (10%)', 'MAR (15%)', 'MNAR (20%)']

for ax, data, title in zip(axes, datasets, titles):
    sns.heatmap(data.isna(), cbar=False, cmap='RdYlGn_r', ax=ax, yticklabels=False)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Features')

axes[0].set_ylabel('Samples')
plt.tight_layout()
plt.show()

## 3. Define Imputation Methods

We'll test methods that are computationally feasible for this comparison.

In [ ]:
# Define methods to compare
methods = {
    'Mean': MeanImputer(),
    'Median': MedianImputer(),
    'KNN-3': KNNImputerMethod(k=3),
    'KNN-5': KNNImputerMethod(k=5),
    'KNN-7': KNNImputerMethod(k=7),
    'MICE': MICEImputer(random_state=42),
    'Hot Deck': HotDeckImputer(random_state=42),
}

print(f"Comparing {len(methods)} imputation methods")

## 4. Run Comparison

Evaluate each method on all three missing data patterns.

In [ ]:
def evaluate_method(method, data_missing, data_complete, method_name):
    """Evaluate a single imputation method."""
    try:
        start_time = time.time()
        imputed = method.impute(data_missing)
        elapsed_time = time.time() - start_time
        
        rmse_score = rmse(data_complete, imputed)
        mae_score = mae(data_complete, imputed)
        
        return {
            'RMSE': rmse_score,
            'MAE': mae_score,
            'Time (s)': elapsed_time,
            'Success': True
        }
    except Exception as e:
        print(f"  ⚠ {method_name} failed: {str(e)[:50]}...")
        return {
            'RMSE': np.nan,
            'MAE': np.nan,
            'Time (s)': np.nan,
            'Success': False
        }

# Run evaluation
results = []

for pattern_name, data_missing in [('MCAR', data_mcar), ('MAR', data_mar), ('MNAR', data_mnar)]:
    print(f"\nEvaluating on {pattern_name} pattern:")
    for method_name, method in methods.items():
        print(f"  Testing {method_name}...", end=' ')
        result = evaluate_method(method, data_missing, data_complete, method_name)
        if result['Success']:
            print(f"✓ RMSE: {result['RMSE']:.4f}, Time: {result['Time (s)']:.3f}s")
        results.append({
            'Pattern': pattern_name,
            'Method': method_name,
            **result
        })

results_df = pd.DataFrame(results)
print("\n✓ Evaluation complete")

## 5. Results Analysis

In [ ]:
# Display results table
print("\n" + "="*80)
print("COMPREHENSIVE RESULTS")
print("="*80)

for pattern in ['MCAR', 'MAR', 'MNAR']:
    print(f"\n{pattern} Pattern:")
    pattern_results = results_df[results_df['Pattern'] == pattern][['Method', 'RMSE', 'MAE', 'Time (s)']]
    pattern_results = pattern_results.sort_values('RMSE')
    print(pattern_results.to_string(index=False))

In [ ]:
# Visualize RMSE comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, pattern in enumerate(['MCAR', 'MAR', 'MNAR']):
    pattern_data = results_df[results_df['Pattern'] == pattern].sort_values('RMSE')
    
    axes[idx].barh(pattern_data['Method'], pattern_data['RMSE'])
    axes[idx].set_xlabel('RMSE', fontsize=11)
    axes[idx].set_title(f'{pattern} Pattern', fontsize=13, fontweight='bold')
    axes[idx].grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, v in enumerate(pattern_data['RMSE']):
        if not np.isnan(v):
            axes[idx].text(v, i, f' {v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.suptitle('RMSE Comparison Across Missing Data Patterns', 
             fontsize=15, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# Visualize computational time
fig, ax = plt.subplots(figsize=(12, 6))

# Pivot data for grouped bar chart
time_pivot = results_df.pivot(index='Method', columns='Pattern', values='Time (s)')
time_pivot = time_pivot.reindex(time_pivot.mean(axis=1).sort_values().index)

time_pivot.plot(kind='bar', ax=ax, width=0.8)
ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title('Computational Time Comparison', fontsize=14, fontweight='bold')
ax.set_xlabel('Method', fontsize=12)
ax.legend(title='Pattern', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Create scatter plot: Accuracy vs Speed
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, pattern in enumerate(['MCAR', 'MAR', 'MNAR']):
    pattern_data = results_df[results_df['Pattern'] == pattern]
    
    axes[idx].scatter(pattern_data['Time (s)'], pattern_data['RMSE'], 
                     s=150, alpha=0.6, c=range(len(pattern_data)))
    
    # Add labels
    for _, row in pattern_data.iterrows():
        if not np.isnan(row['RMSE']):
            axes[idx].annotate(row['Method'], 
                             (row['Time (s)'], row['RMSE']),
                             fontsize=8, ha='left')
    
    axes[idx].set_xlabel('Time (seconds)', fontsize=11)
    axes[idx].set_ylabel('RMSE', fontsize=11)
    axes[idx].set_title(f'{pattern}: Accuracy vs Speed', fontsize=12, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Best Method Analysis

Determine the best method for each pattern based on different criteria.

In [ ]:
print("\n" + "="*80)
print("BEST METHODS BY CRITERIA")
print("="*80)

for pattern in ['MCAR', 'MAR', 'MNAR']:
    pattern_data = results_df[results_df['Pattern'] == pattern]
    
    print(f"\n{pattern} Pattern:")
    print("-" * 40)
    
    # Best accuracy
    best_rmse = pattern_data.loc[pattern_data['RMSE'].idxmin()]
    print(f"Best Accuracy:  {best_rmse['Method']:15s} (RMSE: {best_rmse['RMSE']:.4f})")
    
    # Fastest
    fastest = pattern_data.loc[pattern_data['Time (s)'].idxmin()]
    print(f"Fastest:        {fastest['Method']:15s} (Time: {fastest['Time (s)']:.4f}s)")
    
    # Best balance (normalized score)
    pattern_data_norm = pattern_data.copy()
    pattern_data_norm['RMSE_norm'] = (pattern_data_norm['RMSE'] - pattern_data_norm['RMSE'].min()) / \
                                       (pattern_data_norm['RMSE'].max() - pattern_data_norm['RMSE'].min())
    pattern_data_norm['Time_norm'] = (pattern_data_norm['Time (s)'] - pattern_data_norm['Time (s)'].min()) / \
                                       (pattern_data_norm['Time (s)'].max() - pattern_data_norm['Time (s)'].min())
    pattern_data_norm['Balance'] = pattern_data_norm['RMSE_norm'] + pattern_data_norm['Time_norm']
    best_balance = pattern_data_norm.loc[pattern_data_norm['Balance'].idxmin()]
    print(f"Best Balance:   {best_balance['Method']:15s} (Score: {best_balance['Balance']:.4f})")

## 7. Method Recommendations

Based on the comprehensive evaluation:

### For Speed-Critical Applications:
- **Mean/Median Imputation**: Fastest, acceptable for simple datasets

### For Accuracy-Critical Applications:
- **KNN (k=5 or k=7)**: Best balance of accuracy and reliability
- **MICE**: Best for complex relationships but slower

### For Different Missing Patterns:
- **MCAR**: Most methods perform well; use simple methods for speed
- **MAR**: KNN or MICE recommended
- **MNAR**: More advanced methods (MICE) tend to perform better

### General Guidelines:
1. Start with simple methods (Mean/Median) as baseline
2. Try KNN if computational resources allow
3. Use MICE for complex datasets with time to spare
4. Always validate on your specific dataset

## 8. Next Steps

- Explore **03_real_world_examples.ipynb** for practical applications
- Review **01_getting_started.ipynb** for fundamentals
- Check the [documentation](../README.md) for method details